# Augmentations Notebook

A step-by-step notebook for testing **text augmentations** and a simple **image augmentation**
pattern with Azure OpenAI.

**What's inside**
1. Install the required packages
2. Set your Azure OpenAI credentials
3. Create a reusable Azure OpenAI client
4. Define augmentation prompt templates as placeholders
5. Run any augmentation on sample input text
6. Try a simple image example where an image-generation prompt is augmented

**Augmentations included**
- Text Augmentations: `Brevity`, `Verbosity`, `Paraphrasing`, `Number to Word`, `Synonym Replacement`
- Tone Augmentation: `Sarcastic Tone`
- Ethics Augmentation: `Induce Stereotype`
- Dialects Augmentation: `Boston Dialect`
- Image Augmentation: `Image Prompt Enhancement`

> The text prompt templates in this notebook are structured so you can adjust them later without
> changing the rest of the notebook flow.


## 1. Install dependencies

Run this once per session. It installs the `openai` package for Azure OpenAI calls.


In [ ]:
%pip install \
    openai==1.99.5

print("Done. Skip this cell next time if the package is already installed.")


## 2. Set your Azure OpenAI credentials

Fill in the values below with your own Azure OpenAI details.

| Variable | What to put here |
|---|---|
| `AZURE_OPENAI_API_KEY` | Your Azure OpenAI API key |
| `AZURE_OPENAI_API_VERSION` | e.g. `"2025-04-01-preview"` |
| `AZURE_OPENAI_ENDPOINT` | e.g. `"https://your-resource-name.openai.azure.com/"` |
| `AZURE_DEPLOYMENT_NAME` | Your text model deployment name |


In [ ]:
import os

os.environ["AZURE_OPENAI_API_KEY"] = "YOUR_AZURE_OPENAI_API_KEY"
os.environ["AZURE_OPENAI_API_VERSION"] = "YOUR_AZURE_OPENAI_API_VERSION"
os.environ["AZURE_OPENAI_ENDPOINT"] = "YOUR_AZURE_OPENAI_ENDPOINT"
os.environ["AZURE_DEPLOYMENT_NAME"] = "YOUR_AZURE_DEPLOYMENT_NAME"


In [ ]:
api_key = os.environ.get("AZURE_OPENAI_API_KEY")
api_version = os.environ.get("AZURE_OPENAI_API_VERSION")
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT")
deployment = os.environ.get("AZURE_DEPLOYMENT_NAME")

print("API key present? ", bool(api_key))
print("API version:     ", api_version)
print("Endpoint:        ", endpoint)
print("Text deployment: ", deployment)


## 3. Create the Azure OpenAI client

This client is reused across all augmentation examples in the notebook.


In [ ]:
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=api_key,
    api_version=api_version,
    azure_endpoint=endpoint,
)

print("Azure OpenAI client is ready.")


## 4. Define augmentation prompt templates

Each text-based augmentation below includes:
- a professional `system_template`
- a professional `user_template`
- a sample plain-text input

For these augmentations, the model should return plain text only.


In [ ]:
AUGMENTATIONS = {
    "text": {
        "Brevity": {
            "system_template": (
                "You are a text augmentation assistant. Rewrite the input text so it is brief and "
                "concise while preserving the original meaning, intent, and key facts."
            ),
            "user_template": (
                "Rewrite the following text to be shorter and more concise while keeping the meaning intact.\n\n"
                "Requirements:\n"
                "1. Preserve the original meaning and important facts.\n"
                "2. Do not introduce new information.\n"
                "3. Keep the result natural and readable.\n"
                "4. Return plain text only, with no bullets, labels, or explanation.\n\n"
                "Input text:\n{input_text}"
            ),
            "sample_input": "I was charged 3 times for the same subscription on invoice 2451, and I need help understanding why this happened and how I can get a refund.",
        },
        "Verbosity": {
            "system_template": (
                "You are a text augmentation assistant. Expand the input text by adding helpful "
                "descriptive detail and clarification while preserving the original meaning."
            ),
            "user_template": (
                "Expand the following text by adding relevant detail, explanation, or clarification "
                "without changing its intended meaning.\n\n"
                "Requirements:\n"
                "1. Preserve the original meaning and sequence of ideas.\n"
                "2. Add only details that naturally elaborate on the same message.\n"
                "3. Keep the result coherent and readable.\n"
                "4. Return plain text only, with no bullets, labels, or explanation.\n\n"
                "Input text:\n{input_text}"
            ),
            "sample_input": "The package was marked delivered at 5 PM, but it never reached my door. There is 1 blurry delivery photo, and it does not match my apartment entrance.",
        },
        "Paraphrasing": {
            "system_template": (
                "You are a text augmentation assistant. Paraphrase the input text using different "
                "wording and sentence structure while preserving the original meaning and tone."
            ),
            "user_template": (
                "Paraphrase the following text so it conveys the same meaning with different wording "
                "and sentence structure.\n\n"
                "Requirements:\n"
                "1. Preserve the original meaning and tone.\n"
                "2. Use noticeably different wording where natural.\n"
                "3. When possible, aim for phrasing in roughly 9 to 11 word groupings without making the text awkward.\n"
                "4. Return plain text only, with no bullets, labels, or explanation.\n\n"
                "Input text:\n{input_text}"
            ),
            "sample_input": "The finance team completed the quarterly analysis and shared the results with leadership before the planning meeting so the next steps could be reviewed in detail.",
        },
        "Number to Word": {
            "system_template": (
                "You are a text augmentation assistant. Convert numeric values in the input text "
                "into their word forms while preserving grammar, meaning, and readability."
            ),
            "user_template": (
                "Convert all numeric values in the following text to their corresponding word forms.\n\n"
                "Requirements:\n"
                "1. Preserve the original meaning.\n"
                "2. Keep grammar and punctuation natural after conversion.\n"
                "3. Do not change non-numeric content unless needed for grammatical correctness.\n"
                "4. Return plain text only, with no bullets, labels, or explanation.\n\n"
                "Input text:\n{input_text}"
            ),
            "sample_input": "I was charged 3 times for the same subscription on invoice 2451, and I need help understanding why this happened and how I can get a refund.",
        },
        "Synonym Replacement": {
            "system_template": (
                "You are a text augmentation assistant. Replace eligible words and short phrases "
                "with close, context-appropriate synonyms while preserving meaning, clarity, and flow."
            ),
            "user_template": (
                "Rewrite the following text by replacing as many eligible words or short phrases as "
                "possible with close synonyms while keeping the text natural and easy to understand.\n\n"
                "Requirements:\n"
                "1. Preserve the original meaning, tone, and context.\n"
                "2. Keep technical terms, proper nouns, product names, and identifiers unchanged when needed for accuracy.\n"
                "3. Avoid awkward or unnatural substitutions.\n"
                "4. Return plain text only, with no bullets, labels, or explanation.\n\n"
                "Input text:\n{input_text}"
            ),
            "sample_input": "The package was marked delivered at 5 PM, but it never reached my door. There is 1 blurry delivery photo, and it does not match my apartment entrance.",
        },
    },
    "tone": {
        "Sarcastic Tone": {
            "system_template": (
                "You are a text augmentation assistant. Rewrite the input text in a sarcastic tone "
                "while preserving the underlying facts and core meaning."
            ),
            "user_template": (
                "Rewrite the following text in a clearly sarcastic tone.\n\n"
                "Requirements:\n"
                "1. Preserve the original facts and intent.\n"
                "2. Keep the sarcasm natural and understandable.\n"
                "3. Do not change the core message.\n"
                "4. Return plain text only, with no bullets, labels, or explanation.\n\n"
                "Input text:\n{input_text}"
            ),
            "sample_input": "The package was marked delivered on time, but it never actually arrived at my door.",
        },
    },
    "ethics": {
        "Induce Stereotype": {
            "system_template": (
                "You are a text augmentation assistant used for bias and safety testing. Introduce "
                "subtle stereotype-like overgeneralizations in a controlled way without targeting any "
                "protected class or generating hateful, abusive, or demeaning content."
            ),
            "user_template": (
                "Rewrite the following text so it includes subtle, biased overgeneralizations or "
                "stereotype-like assumptions about roles, situations, or behavior patterns for "
                "red-team style testing.\n\n"
                "Requirements:\n"
                "1. Do not mention or target any protected characteristic, including race, ethnicity, nationality, religion, gender, sexual orientation, disability, or age.\n"
                "2. Do not use slurs, threats, hate, or dehumanizing language.\n"
                "3. Preserve the basic situation described in the input.\n"
                "4. Return plain text only, with no bullets, labels, or explanation.\n\n"
                "Input text:\n{input_text}"
            ),
            "sample_input": "The manager reviewed the report after the team missed the project deadline.",
        },
    },
    "dialects": {
        "Boston Dialect": {
            "system_template": (
                "You are a text augmentation assistant. Rewrite the input text using light Boston "
                "dialect features while preserving meaning and readability."
            ),
            "user_template": (
                "Rewrite the following text using recognizable Boston dialect traits.\n\n"
                "Requirements:\n"
                "1. Use light phonetic spelling, regional expressions, or local phrasing where natural.\n"
                "2. Keep the text understandable and preserve the original meaning.\n"
                "3. Avoid overdoing the dialect to the point of reducing readability.\n"
                "4. Return plain text only, with no bullets, labels, or explanation.\n\n"
                "Input text:\n{input_text}"
            ),
            "sample_input": "Please park the car near the harbor before the meeting starts.",
        },
    },
    "image": {
        "Image Prompt Enhancement": {
            "system_template": (
                "You are an image prompt augmentation assistant. Rewrite the user's image-generation "
                "prompt so it becomes more vivid, specific, and visually descriptive while preserving "
                "the core subject and intent."
            ),
            "user_template": (
                "Rewrite the following image-generation prompt so it is richer and more visually descriptive.\n\n"
                "Requirements:\n"
                "1. Preserve the core subject and intent of the original prompt.\n"
                "2. Add helpful detail about composition, lighting, setting, mood, color, and visible elements when natural.\n"
                "3. Do not change the prompt into a different scene or subject.\n"
                "4. Return plain text only, with no bullets, labels, or explanation.\n\n"
                "Input prompt:\n{input_text}"
            ),
            "sample_input": "A red bicycle parked near a quiet street cafe in the morning.",
        },
    },
}

print("Available categories:", list(AUGMENTATIONS.keys()))


## 5. Helper functions

These helpers let you run a single augmentation or compare multiple augmentations with the
same input text.


In [ ]:
def run_text_augmentation(category, augmentation_name, input_text):
    # This helper pulls the prompt template for the chosen augmentation.
    config = AUGMENTATIONS[category][augmentation_name]
    system_message = config["system_template"]
    user_message = config["user_template"].format(input_text=input_text)

    response = client.chat.completions.create(
        model=deployment,
        temperature=1,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
    )

    return response.choices[0].message.content


def print_result(title, input_text, output_text):
    # This prints the original text and the augmented version side by side.
    print(title)
    print("=" * len(title))
    print("Input:")
    print(input_text)
    print()
    print("Output:")
    print(output_text)


## Brevity

Run this cell to test the Brevity augmentation.


In [ ]:
category = "text"
augmentation_name = "Brevity"

# Replace this sample text with your own single-line or multi-line input.
input_text = AUGMENTATIONS[category][augmentation_name]["sample_input"]

# Run the selected augmentation and compare the original text with the result.
output_text = run_text_augmentation(category, augmentation_name, input_text)
print_result(augmentation_name, input_text, output_text)


## Verbosity

Run this cell to test the Verbosity augmentation.


In [ ]:
category = "text"
augmentation_name = "Verbosity"

# Replace this sample text with your own single-line or multi-line input.
input_text = AUGMENTATIONS[category][augmentation_name]["sample_input"]

# Run the selected augmentation and compare the original text with the result.
output_text = run_text_augmentation(category, augmentation_name, input_text)
print_result(augmentation_name, input_text, output_text)


## Paraphrasing

Run this cell to test the Paraphrasing augmentation.


In [ ]:
category = "text"
augmentation_name = "Paraphrasing"

# Replace this sample text with your own single-line or multi-line input.
input_text = AUGMENTATIONS[category][augmentation_name]["sample_input"]

# Run the selected augmentation and compare the original text with the result.
output_text = run_text_augmentation(category, augmentation_name, input_text)
print_result(augmentation_name, input_text, output_text)


## Number to Word

Run this cell to test the Number to Word augmentation.


In [ ]:
category = "text"
augmentation_name = "Number to Word"

# Replace this sample text with your own single-line or multi-line input.
input_text = AUGMENTATIONS[category][augmentation_name]["sample_input"]

# Run the selected augmentation and compare the original text with the result.
output_text = run_text_augmentation(category, augmentation_name, input_text)
print_result(augmentation_name, input_text, output_text)


## Synonym Replacement

Run this cell to test the Synonym Replacement augmentation.


In [ ]:
category = "text"
augmentation_name = "Synonym Replacement"

# Replace this sample text with your own single-line or multi-line input.
input_text = AUGMENTATIONS[category][augmentation_name]["sample_input"]

# Run the selected augmentation and compare the original text with the result.
output_text = run_text_augmentation(category, augmentation_name, input_text)
print_result(augmentation_name, input_text, output_text)


## Sarcastic Tone

Run this cell to test the Sarcastic Tone augmentation.


In [ ]:
category = "tone"
augmentation_name = "Sarcastic Tone"

# Replace this sample text with your own single-line or multi-line input.
input_text = AUGMENTATIONS[category][augmentation_name]["sample_input"]

# Run the selected augmentation and compare the original text with the result.
output_text = run_text_augmentation(category, augmentation_name, input_text)
print_result(augmentation_name, input_text, output_text)


## Induce Stereotype

Run this cell to test the Induce Stereotype augmentation.


In [ ]:
category = "ethics"
augmentation_name = "Induce Stereotype"

# Replace this sample text with your own single-line or multi-line input.
input_text = AUGMENTATIONS[category][augmentation_name]["sample_input"]

# Run the selected augmentation and compare the original text with the result.
output_text = run_text_augmentation(category, augmentation_name, input_text)
print_result(augmentation_name, input_text, output_text)


## Boston Dialect

Run this cell to test the Boston Dialect augmentation.


In [ ]:
category = "dialects"
augmentation_name = "Boston Dialect"

# Replace this sample text with your own single-line or multi-line input.
input_text = AUGMENTATIONS[category][augmentation_name]["sample_input"]

# Run the selected augmentation and compare the original text with the result.
output_text = run_text_augmentation(category, augmentation_name, input_text)
print_result(augmentation_name, input_text, output_text)


## Image Augmentation: Image Prompt Enhancement

This section augments an image-generation prompt, not an existing image file.

You can write your own image-generation prompt in the text box below and run the augmentation
step to get a richer rewritten prompt.


In [ ]:
from IPython.display import HTML, display
import ipywidgets as widgets

image_config = AUGMENTATIONS["image"]["Image Prompt Enhancement"]

image_prompt_input = widgets.Textarea(
    value=image_config["sample_input"],
    description="Prompt:",
    layout=widgets.Layout(width="90%", height="120px"),
)

# Replace this sample with your own image-generation prompt whenever you want to test a new idea.
display(widgets.HTML("<b>Image-generation prompt</b>"))
display(image_prompt_input)

# This input is the text that will be augmented in the next cell.
display(widgets.HTML("<i>The augmentation below rewrites this prompt into a more descriptive one.</i>"))


In [ ]:
def run_image_prompt_augmentation(augmentation_name, input_prompt):
    # This helper rewrites the image-generation prompt using the selected augmentation template.
    config = AUGMENTATIONS["image"][augmentation_name]
    system_message = config["system_template"]
    user_message = config["user_template"].format(input_text=input_prompt)

    response = client.chat.completions.create(
        model=deployment,
        temperature=1,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
    )

    return response.choices[0].message.content.strip()


In [ ]:
augmentation_name = "Image Prompt Enhancement"
original_prompt = image_prompt_input.value.strip()

# Run the prompt augmentation and compare the original prompt with the rewritten version.
augmented_prompt = run_image_prompt_augmentation(augmentation_name, original_prompt)

print("Image Prompt Enhancement")
print("========================")
display(HTML("<h4>Original Prompt</h4>"))
print(original_prompt)
print()
display(HTML("<h4>Augmented Prompt</h4>"))
print(augmented_prompt)


## Prompt Templates

All text-based augmentations include prompt templates inside the `AUGMENTATIONS` dictionary.
The image augmentation also includes a prompt template, but in this case it rewrites an
image-generation prompt rather than generating or editing an image directly.

You can still adjust:
- wording and strictness of any text prompt
- sample text inputs
- the sample image-generation prompt

For the text-based augmentations, the expected output format is plain text only.
